<a href="https://colab.research.google.com/github/JasonL888/AI_Experiments/blob/main/QuizGenerator/QuizGenerator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# install pre-reqs

In [39]:
!pip install -qU langchain-classic langchain-community langchain-huggingface
# can safely ignore the pip dependency error about google-colab 1.0.0 requires requests==2.32.4 but you have requests 2.32.5

In [40]:
!pip install -qU faiss-cpu pypdf transformers accelerate bitsandbytes fpdf2

# Import Python Modules

In [41]:
import os
import torch
from google.colab import files
from fpdf import FPDF
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig

# Upload PDF

In [42]:
print("Please upload your PDF file:")
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

Please upload your PDF file:


IndexError: list index out of range

# Preprocess PDF

In [43]:
loader = PyPDFLoader(pdf_filename)
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# Local Embeddings

In [44]:
# This model is only ~80MB and runs instantly on CPU/GPU
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Load Quantized LLM

In [45]:
model_id = "microsoft/Phi-3.5-mini-instruct"

In [46]:
# 4-bit quantization allows the model to fit in < 3GB of VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

In [47]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [48]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [49]:
# Create a local pipeline for LangChain
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    return_full_text=False
)

Device set to use cuda:0


In [50]:
llm = HuggingFacePipeline(pipeline=hf_pipeline)

# RAG Chain Setup

In [90]:
system_prompt = (
    "You are a quiz master. Your task is to generate exactly 5 multiple-choice questions "
    "based *only* on the provided context. Each question must have 4 options (a, b, c, d). "
    "For each question, clearly state the correct answer immediately after the options in the format 'Answer X)'"
    "Your output MUST start directly with '1.' and contain only the quiz questions and answers in the specified format:"
    "\n\n[question number]. [question text]"
    "\n\ta. [option a text]"
    "\n\tb. [option b text]"
    "\n\tc. [option c text]"
    "\n\td. [option d text]"
    "\nAnswer: [correct option character a, b, c or d]"
    "\n\n"
    "Absolutely no introductory text, headings like 'Quiz Questions:', or concluding sentences should be included. Do not provide any '### Answer' at the end"
    "If the context is insufficient to create 5 questions, create fewer or state it explicitly."
    "\n\nContext: {context}"
)

In [91]:
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Generate a quiz from the uploaded document."),
])

In [92]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)

In [93]:
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

# Execute

In [94]:
print("\nGenerating Quiz... (This may take a moment on Colab)\n")
response = rag_chain.invoke({"input": "Generate 5 questions"})


Generating Quiz... (This may take a moment on Colab)



In [95]:
response

{'input': 'Generate 5 questions',
 'context': [Document(id='4a520465-c55a-4c2e-b532-905444ffc536', metadata={'producer': 'Microsoft® PowerPoint® 2019', 'creator': 'Microsoft® PowerPoint® 2019', 'creationdate': '2025-01-03T18:46:25+08:00', 'title': 'PowerPoint Presentation', 'author': 'Suraj Uprety - Content Marketing Manager', 'moddate': '2025-01-03T18:46:25+08:00', 'source': 'IU6_-_Agile_Continuous_Improvement_and_Growth.pdf', 'total_pages': 22, 'page': 16, 'page_label': '17'}, page_content='Sprint Planning and Review\nDemonstration of Work Completed\n• The team demonstrates the work completed during the sprint.\n• This includes showcasing new features, improvements, or any \nother deliverables.\nSprint Review Process\n• Stakeholders provide feedback during the review.\n• This feedback is valuable for continuous improvement and \nensures that the end product aligns with stakeholder \nexpectations.\nSprint Review Process\nBacklog Refinement\n• Before sprint planning, the team refines t

In [99]:
quiz_text = response["answer"]

# Post-processing to ensure desired format
# Remove any leading/trailing whitespace
quiz_text = quiz_text.strip()

# Find the index of the first question '1.'
quiz_start_index = quiz_text.find("1.")

# If '1.' is found, slice the string from there
if quiz_start_index != -1:
    quiz_text = quiz_text[quiz_start_index:]
else:
    print("Warning: Could not find the start of the quiz (text '1.'). Output might be malformed.")

# Replace 'Answer: ' with 'Answer ' to remove the colon, if present
#quiz_text = quiz_text.replace("Answer: ", "Answer ")

print(quiz_text)

1. What is the primary purpose of the Sprint Review Process?
a. To assign new tasks to the team
b. To provide feedback and ensure alignment with stakeholder expectations
c. To evaluate the performance of individual team members
d. To finalize the product for release
Answer: b

2. What does Capacity Planning involve in the context of Sprint Planning?
a. Determining the team's capacity and historical velocity
b. Calculating the total cost of the project
c. Assigning tasks based on employee availability
d. Planning the company's overall workload
Answer: a

3. What is a key benefit of Iterative Development in Scrum?
a. It allows for large, comprehensive releases at once
b. It ensures the project remains aligned with the evolving landscape
c. It eliminates the need for a product backlog
d. It reduces the importance of stakeholder feedback
Answer: b

4. What does the team demonstrate during the Sprint Review?
a. The work completed during the sprint
b. The final product ready for release
c. T

In [100]:
import re

# Split the quiz_text into lines
lines = quiz_text.split('\n')

# Filter out lines that start with 'Human:' or 'Assistant:'
filtered_lines = [line for line in lines if not re.match(r'^(Human:|Assistant:)', line.strip())]

# Join the filtered lines back together
quiz_text = '\n'.join(filtered_lines).strip()

print(quiz_text)

1. What is the primary purpose of the Sprint Review Process?
a. To assign new tasks to the team
b. To provide feedback and ensure alignment with stakeholder expectations
c. To evaluate the performance of individual team members
d. To finalize the product for release
Answer: b

2. What does Capacity Planning involve in the context of Sprint Planning?
a. Determining the team's capacity and historical velocity
b. Calculating the total cost of the project
c. Assigning tasks based on employee availability
d. Planning the company's overall workload
Answer: a

3. What is a key benefit of Iterative Development in Scrum?
a. It allows for large, comprehensive releases at once
b. It ensures the project remains aligned with the evolving landscape
c. It eliminates the need for a product backlog
d. It reduces the importance of stakeholder feedback
Answer: b

4. What does the team demonstrate during the Sprint Review?
a. The work completed during the sprint
b. The final product ready for release
c. T

# Export to PDF

In [101]:
def save_quiz_to_pdf(text, filename="generated_quiz.pdf"):
    pdf = FPDF()
    pdf.add_page()

    # Use a built-in font that handles basic formatting
    pdf.set_font("Helvetica", size=12)

    # Title
    pdf.set_font("Helvetica", style="B", size=16)
    pdf.cell(200, 10, txt="AI-Generated Quiz", ln=True, align='C')
    pdf.ln(10) # Line break

    # Content
    pdf.set_font("Helvetica", size=11)
    # multi_cell handles line wrapping automatically
    pdf.multi_cell(0, 8, txt=text)

    pdf.output(filename)
    print(f"PDF created: {filename}")
    files.download(filename) # Triggers browser download

In [102]:
save_quiz_to_pdf(quiz_text)

PDF created: generated_quiz.pdf


/tmp/ipython-input-1190046463.py:10: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.cell(200, 10, txt="AI-Generated Quiz", ln=True, align='C')
/tmp/ipython-input-1190046463.py:10: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(200, 10, txt="AI-Generated Quiz", ln=True, align='C')
/tmp/ipython-input-1190046463.py:16: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.multi_cell(0, 8, txt=text)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>